# Imports and loading

In [1]:
from pathlib import Path
import json
import sys
import os
from typing import Any, Dict, List, Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login, whoami
from getpass import getpass

REPO = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine")
sys.path.append(str(REPO))

from evaluate.extract_answer import extract_boxed_answer, is_equiv

In [3]:
os.environ["HF_TOKEN"] = getpass("Paste your HF token: ")
login(token=os.environ["HF_TOKEN"])
whoami(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'type': 'user',
 'id': '666b03c6d5e7ca261c77c468',
 'name': 'emma-anastassova',
 'fullname': 'Emma Anastassova',
 'isPro': False,
 'avatarUrl': '/avatars/1d2e123afae29b0650ead4c462dd8a00.svg',
 'orgs': [{'type': 'org',
   'id': '69f9bb793215252d1c045bcc',
   'name': 'mystery-machine',
   'fullname': 'Mystery Machine',
   'email': 'hugo.demulerey@gmail.com',
   'canPay': False,
   'billingMode': 'prepaid',
   'periodEnd': 1780272000,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/69efbeea1eee727f224ffea0/HSXYTmoMqSLuiimzrxysQ.webp',
   'roleInOrg': 'write'},
  {'type': 'org',
   'id': '69fb87d3a5fe2c0acc1cba16',
   'name': 'cs-552-2026-mystery-machine',
   'fullname': 'Mystery Machine',
   'email': 'emma.anastassova@gmail.com',
   'canPay': False,
   'billingMode': 'prepaid',
   'periodEnd': 1780272000,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/666b03c6d5e7ca261c77c468/92GKgofkeN67eY6Wz5qGU.webp',
   'roleInOrg': 'admin'}],
 'a

In [4]:
from huggingface_hub import repo_info

repo_info(
    repo_id="cs-552-2026-mystery-machine/group_model",
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)

ModelInfo(id='cs-552-2026-mystery-machine/group_model', author='cs-552-2026-mystery-machine', base_models=None, card_data={'base_model': ['Qwen/Qwen3-1.7B'], 'datasets': None, 'eval_results': None, 'language': ['en', 'ru', 'hi', 'it', 'es', 'zh'], 'library_name': None, 'license': None, 'license_name': None, 'license_link': None, 'metrics': None, 'model_name': None, 'pipeline_tag': None, 'tags': None}, children_model_count=None, config={}, created_at=datetime.datetime(2026, 5, 6, 18, 41, 30, tzinfo=datetime.timezone.utc), disabled=False, downloads=0, downloads_all_time=None, eval_results=None, gated=False, gguf=None, inference=None, inference_provider_mapping=None, last_modified=datetime.datetime(2026, 5, 6, 18, 44, 55, tzinfo=datetime.timezone.utc), library_name=None, likes=0, mask_token=None, model_index=None, pipeline_tag=None, private=False, resource_group=None, safetensors=None, security_repo_status=None, sha='de8396bf533e9879c5c7fff4711160a98594c5f5', siblings=[RepoSibling(rfilena

In [67]:
RESULTS_DIR = REPO / "multilingual" / "results" / "base_qwen"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3-1.7B"

TARGET_LANGUAGES = {
    "russian": {
        "include_config": "Russian",
        "mgsm_configs": ["ru"],
    },
    "chinese": {
        "include_config": "Chinese",
        "mgsm_configs": ["zh"],
    },
    "italian": {
        "include_config": "Italian",
        "mgsm_configs": [],
    },
    "spanish": {
        "include_config": "Spanish",
        "mgsm_configs": ["es"],
    },
    "hindi": {
        "include_config": "Hindi",
        "mgsm_configs": [],
    },
    "english": {
        "include_config": None,
        "mgsm_configs": ["en"],
    },
}

N_PER_LANGUAGE = 5 # just to ttest
MAX_NEW_TOKENS_BY_TASK = {
    "mcq": 128,
    "math": 1024,
}

LETTERS = [chr(ord("A") + i) for i in range(26)]
MCQ_LETTERS = ["A", "B", "C", "D"]

In [6]:
hf_token = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [73]:
# generation
def build_messages(prompt: str, task_type: str) -> List[Dict[str, str]]:
    if task_type == "mcq":
        system = (
            "You are solving a multiple-choice question. "
            "You must output exactly one final answer in the format \\boxed{X}, where X is the letter of the correct option. "
            "Do not output the option text. Do not output explanations. "
            "Do not output anything after the boxed answer. /no_think"
        )

    elif task_type == "math":
        system = (
            "You are a multilingual mathematical reasoning assistant. "
            "Solve the problem step by step. Put the final numeric answer inside \\boxed{...}."
            "Do not put units or explanations inside the box."
        )

    else:
        system = (
            "You are a multilingual evaluation assistant. "
            "Answer the question. "
            "At the end, put the final answer inside \\boxed{...}."
        )

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

@torch.inference_mode()
@torch.inference_mode()
def generate_answer(prompt: str, task_type: str, max_new_tokens: int) -> Dict[str, Any]:
    messages = build_messages(prompt, task_type)

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated, skip_special_tokens=False)

    return {
        "text": text,
        "n_generated_tokens": generated.shape[-1],
        "hit_max_tokens": generated.shape[-1] >= max_new_tokens,
    }

In [33]:
# helpers
def pick_split(ds) -> str:
    for split in ["test", "validation", "dev", "train"]:
        if split in ds:
            return split
    return list(ds.keys())[0]


def normalize_mc_answer(x: Any) -> Optional[str]:
    if x is None:
        return None
    return str(x).strip().upper()


def normalize_text_answer(x: Any) -> Optional[str]:
    if x is None:
        return None
    return str(x).strip()


def select_at_most(dataset, n=None):
    dataset = dataset.shuffle(seed=42)
    if n is None:
        return dataset
    n_eff = min(n, len(dataset))
    return dataset.select(range(n_eff))


def save_jsonl(df: pd.DataFrame, dataset_name: str, language: str) -> Path:
    out_path = RESULTS_DIR / f"{dataset_name}_{language}_base_qwen.jsonl"
    df.to_json(out_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {out_path}")
    return out_path


# Dataset loading

In [23]:
def pick_split(ds) -> str:
    for split in ["test", "validation", "dev", "train"]:
        if split in ds:
            return split
    return list(ds.keys())[0]


def select_at_most(dataset, n=None):
    dataset = dataset.shuffle(seed=42)
    if n is None:
        return dataset
    n_eff = min(n, len(dataset))
    return dataset.select(range(n_eff))

### INCLUDE formatting

In [59]:
def detect_include_language_field(dataset) -> str:
    for candidate in ["language", "lang", "locale"]:
        if candidate in dataset.features:
            return candidate
    raise KeyError(f"No language field found. Features: {dataset.features}")


def format_include_example(ex: Dict[str, Any], language: str, idx: int) -> Dict[str, Any]:
    question = str(ex["question"])

    choices = [
        str(ex["option_a"]),
        str(ex["option_b"]),
        str(ex["option_c"]),
        str(ex["option_d"]),
    ]

    options = "\n".join(
        f"{LETTERS[i]}) {choice}"
        for i, choice in enumerate(choices)
    )

    prompt = (
        f"{question}\n\n"
        f"{options}\n\n"
        "Choose the correct option. "
        "Return exactly one answer in this format: \\boxed{X}, replacing X by the correct option letter. "
        "Do not write the option text. Do not write an explanation."
    )

    gold = LETTERS[int(ex["answer"])]

    metadata = {
        "language": ex["language"],
        "country": ex["country"],
        "domain": ex["domain"],
        "subject": ex["subject"],
        "regional_feature": ex["regional_feature"],
        "level": ex["level"],
    }

    return {
        "id": f"include_{language}_{idx}",
        "dataset": "include",
        "language": language,
        "task_type": "mcq",
        "prompt": prompt,
        "answer": gold,
        "metadata": metadata,
    }


def load_include_examples(language: str, n: int) -> List[Dict[str, Any]]:
    include_config = TARGET_LANGUAGES[language]["include_config"]

    if include_config is None:
        print(f"[INCLUDE] No INCLUDE config for {language}; skipping.")
        return []

    ds = load_dataset("CohereLabs/include-base-44", include_config)

    # Use test for baseline evaluation, not validation.
    split = "test" if "test" in ds else pick_split(ds)
    data = ds[split]

    if len(data) == 0:
        print(f"[INCLUDE] Empty split for {language}.")
        return []

    sampled = select_at_most(data, n)

    return [
        format_include_example(sampled[i], language, i)
        for i in range(len(sampled))
    ]

### MGSM formatting

In [60]:
def infer_mgsm_question_key(ex: Dict[str, Any]) -> str:
    for key in ["question", "problem", "input", "query", "prompt"]:
        if key in ex:
            return key
    raise KeyError(f"Could not infer MGSM question key. Keys: {list(ex.keys())}")


def infer_mgsm_answer_key(ex: Dict[str, Any]) -> str:
    for key in ["answer_number", "answer", "target", "output", "label"]:
        if key in ex and ex[key] is not None:
            return key
    raise KeyError(f"Could not infer MGSM answer key. Keys: {list(ex.keys())}")


def normalize_mgsm_gold_answer(answer: Any) -> str:
    answer_str = str(answer).strip()

    if "####" in answer_str:
        answer_str = answer_str.split("####")[-1].strip()

    return answer_str


def format_mgsm_example(ex: Dict[str, Any], language: str, idx: int) -> Dict[str, Any]:
    question_key = infer_mgsm_question_key(ex)
    answer_key = infer_mgsm_answer_key(ex)

    question = str(ex[question_key])
    gold = normalize_mgsm_gold_answer(ex[answer_key])

    prompt = (
        f"{question}\n\n"
        "Solve the problem. Put only the final numeric answer inside \\boxed{...}."
    )

    metadata = {
        k: ex[k]
        for k in ex.keys()
        if k not in {question_key, answer_key}
    }

    return {
        "id": f"mgsm_{language}_{idx}",
        "dataset": "mgsm",
        "language": language,
        "task_type": "math",
        "prompt": prompt,
        "answer": gold,
        "metadata": metadata,
    }
def load_mgsm_examples(language: str, n: int) -> List[Dict[str, Any]]:
    mgsm_configs = TARGET_LANGUAGES[language]["mgsm_configs"]

    if not mgsm_configs:
        print(f"[MGSM] No MGSM config for {language}; skipping.")
        return []

    available_configs = get_dataset_config_names("juletxara/mgsm")
    config_lookup = {c.lower(): c for c in available_configs}

    chosen_config = None
    for candidate in mgsm_configs:
        if candidate.lower() in config_lookup:
            chosen_config = config_lookup[candidate.lower()]
            break

    if chosen_config is None:
        print(f"[MGSM] No config found for {language}. Available configs: {available_configs}")
        return []

    ds = load_dataset("juletxara/mgsm", chosen_config)
    split = pick_split(ds)
    data = ds[split]

    if len(data) == 0:
        print(f"[MGSM] Empty split for {language}.")
        return []

    sampled = select_at_most(data, n)

    return [
        format_mgsm_example(sampled[i], language, i)
        for i in range(len(sampled))
    ]

# Evaluation
Just to test on 5 examples before the full run

In [ ]:
import re
def extract_mcq_prediction_with_fallback(output: str) -> Optional[str]:
    """
    MCQ extractor. NOT the official course parser, 
    only useful for determining whether the model knew the answer but failed the boxed-output format.
    Priority:
    1. Official boxed answer: \\boxed{A}
    2. Explicit answer patterns: Answer: A, 答案：A, Risposta: A, etc.
    3. Option-style patterns: A), A., A:
    4. Single standalone letter: A
    """
    boxed = extract_boxed_answer(output)
    if boxed is not None:
        boxed = boxed.strip().upper()
        if boxed in MCQ_LETTERS:
            return boxed

    text = output.strip()

    # remove Qwen thinking block if present.
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    # remove common special tokens
    text = text.replace("<|im_end|>", "").strip()
    text = text.replace("<|endoftext|>", "").strip()

    # Normalize whitespace.
    text = re.sub(r"\s+", " ", text).strip()

    patterns = [
        # explicit answer markers in several languages
        r"(?:answer|final answer|correct answer|risposta|respuesta|réponse|antwort|ответ|答案|答)\s*[:：]?\s*([A-Z])\b",

        # box-like but malformed
        r"\\boxed\s*\{?\s*([A-Z])\s*\}?",

        # option format: A), A., A:, A：
        r"\b([A-Z])\s*[\)\.:：]",

        # starts directly with option letter followed by text.
        r"^\s*([A-Z])\b",

        # just a single letter.
        r"^\s*([A-Z])\s*$",
    ]

    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            letter = match.group(1).upper()
            if letter in MCQ_LETTERS:
                return letter

    return None

In [74]:
def score_prediction(prediction: Optional[str], gold: str, task_type: str) -> bool:
    if prediction is None:
        return False

    if task_type == "mcq":
        return normalize_mc_answer(prediction) == normalize_mc_answer(gold)

    if task_type == "math":
        return is_equiv(prediction, gold)

    return normalize_text_answer(prediction) == normalize_text_answer(gold)


def evaluate_examples(
    examples: List[Dict[str, Any]],
    dataset_name: str,
    language: str,
) -> pd.DataFrame:
    rows = []

    for ex in tqdm(examples, desc=f"{dataset_name}/{language}"):
        max_new_tokens = MAX_NEW_TOKENS_BY_TASK[ex["task_type"]]

        gen = generate_answer(
            prompt=ex["prompt"],
            task_type=ex["task_type"],
            max_new_tokens=max_new_tokens,
        )

        output = gen["text"]

        # Strict official-style prediction: only boxed answers count.
        prediction = extract_boxed_answer(output)
        missing_box = prediction is None
        correct = score_prediction(prediction, ex["answer"], ex["task_type"])

        # fallback prediction for diagnostics, MCQ only, extract answer even if not boxed
        if ex["task_type"] == "mcq":
            fallback_prediction = extract_mcq_prediction_with_fallback(output)
        else:
            fallback_prediction = prediction

        fallback_correct = score_prediction(
            fallback_prediction,
            ex["answer"],
            ex["task_type"],
        )

        rows.append({
            "id": ex["id"],
            "dataset": dataset_name,
            "language": language,
            "task_type": ex["task_type"],
            "gold": ex["answer"],

            # Official-style fields
            "prediction": prediction,
            "correct": correct,
            "missing_box": missing_box,

            # Diagnostic fields
            "fallback_prediction": fallback_prediction,
            "fallback_correct": fallback_correct,

            # Generation diagnostics
            "n_generated_tokens": gen["n_generated_tokens"],
            "hit_max_tokens": gen["hit_max_tokens"],
            "max_new_tokens": max_new_tokens,

            # Raw data
            "prompt": ex["prompt"],
            "output": output,
            "metadata": json.dumps(ex.get("metadata", {}), ensure_ascii=False),
        })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        n_correct = int(df["correct"].sum())
        n_total = len(df)

        strict_acc = n_correct / n_total
        fallback_acc = df["fallback_correct"].mean()
        box_compliance = (~df["missing_box"]).mean()
        max_token_rate = df["hit_max_tokens"].mean()

        valid = df[~df["missing_box"]]
        valid_only_acc = valid["correct"].mean() if len(valid) > 0 else None

        print(
            f"{dataset_name}/{language}: "
            f"strict_acc = {strict_acc:.3f} ({n_correct}/{n_total}), "
            f"box_compliance = {box_compliance:.3f}, "
            f"valid_only_acc = {valid_only_acc if valid_only_acc is not None else 'NA'}, "
            f"fallback_acc = {fallback_acc:.3f}, "
            f"hit_max_tokens = {max_token_rate:.3f}"
        )
    else:
        print(f"{dataset_name}/{language}: no examples")

    return df

In [68]:
all_results = []

for language in TARGET_LANGUAGES:
    include_examples = load_include_examples(language, N_PER_LANGUAGE)

    if len(include_examples) > 0:
        df_include = evaluate_examples(
            include_examples,
            dataset_name="include",
            language=language,
        )
        save_jsonl(df_include, "include", language)
        all_results.append(df_include)

    mgsm_examples = load_mgsm_examples(language, N_PER_LANGUAGE)

    if len(mgsm_examples) > 0:
        df_mgsm = evaluate_examples(
            mgsm_examples,
            dataset_name="mgsm",
            language=language,
        )
        save_jsonl(df_mgsm, "mgsm", language)
        all_results.append(df_mgsm)

include/russian:   0%|          | 0/5 [00:00<?, ?it/s]

include/russian: strict_acc = 1.000 (5/5), fallback_acc = 1.000, box_compliance = 1.000
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/include_russian_base_qwen.jsonl


mgsm/russian:   0%|          | 0/5 [00:00<?, ?it/s]

mgsm/russian: strict_acc = 0.600 (3/5), fallback_acc = 0.600, box_compliance = 0.600
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/mgsm_russian_base_qwen.jsonl


include/chinese:   0%|          | 0/5 [00:00<?, ?it/s]

...

mgsm/spanish: strict_acc = 0.800 (4/5), fallback_acc = 0.800, box_compliance = 0.800
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/mgsm_spanish_base_qwen.jsonl


include/hindi:   0%|          | 0/5 [00:00<?, ?it/s]

include/hindi: strict_acc = 0.400 (2/5), fallback_acc = 0.400, box_compliance = 1.000
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/include_hindi_base_qwen.jsonl
[MGSM] No MGSM config for hindi; skipping.
[INCLUDE] No INCLUDE config for english; skipping.


mgsm/english:   0%|          | 0/5 [00:00<?, ?it/s]

mgsm/english: strict_acc = 0.600 (3/5), fallback_acc = 0.600, box_compliance = 0.600
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/mgsm_english_base_qwen.jsonl


In [69]:
df_all = pd.concat(all_results, ignore_index=True)

summary_rows = []

for (dataset, language), group in df_all.groupby(["dataset", "language"]):
    valid = group[~group["missing_box"]]

    summary_rows.append({
        "dataset": dataset,
        "language": language,
        "n": len(group),
        "strict_accuracy": group["correct"].mean(),
        "box_compliance_rate": (~group["missing_box"]).mean(),
        "valid_only_accuracy": valid["correct"].mean() if len(valid) > 0 else None,
        "fallback_accuracy": group["fallback_correct"].mean(),
        "n_valid_boxed": len(valid),
    })

summary = pd.DataFrame(summary_rows)
summary

,dataset,language,n,strict_accuracy,box_compliance_rate,valid_only_accuracy,fallback_accuracy,n_valid_boxed
0,include,chinese,5,0.2,1.0,0.20,0.2,5
1,include,hindi,5,0.4,1.0,0.40,0.4,5
2,include,italian,5,0.6,1.0,0.60,0.6,5
3,include,russian,5,1.0,1.0,1.00,1.0,5
4,include,spanish,5,0.2,1.0,0.20,0.2,5
5,mgsm,chinese,5,0.6,0.8,0.75,0.6,4
6,mgsm,english,5,0.6,0.6,1.00,0.6,3
7,mgsm,russian,5,0.6,0.6,1.00,0.6,3
8,mgsm,spanish,5,0.8,0.8,1.00,0.8,4


# length of the datasets

In [70]:
include_examples = load_include_examples(language, None)

In [71]:
mgsm_examples = load_mgsm_examples(language, None)

In [72]:
len(include_examples), len(mgsm_examples)

(550, 250)